# FB15K baseline: RGAT + ComplEx
Existing two-layer baseline, seed0, width100 (50 complex dimensions), 10 negatives,
300 epochs, no warmup. Effective batch32768 = micro4096 × 8: 15 updates/epoch, 4500 total.
Validation every10 epochs; select by validation MRR and test only the best checkpoint.
This is not pure ComplEx paper reproduction. No RoBERTa/text is loaded.
Only section5 launches training. Full GPU memory/runtime capacity must be measured.


## 1. GPU runtime


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_CACHE'] = '/content/hf_cache/hub'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/content/hf_cache/hub'
os.environ.pop('TRANSFORMERS_CACHE', None)
import torch
print('Python:', sys.version, '; PyTorch:', torch.__version__, '; CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('Total GPU GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
print('Free / total bytes:', torch.cuda.mem_get_info())
print('Keep GPU, packages and numerical policy consistent across resumed sessions.')

torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


## 2. Fetch main, mount Drive and install dependencies


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SHARED_ROOT = Path('/content/drive/MyDrive/Research/scratch_pipeline_v1')
STAGE_ROOT = Path('/content/drive/MyDrive/Research/fb15k_baseline_300')
RUN_TAG = 'fb15k_baseline_300_seed0_accum8_v2_run1'  # Use the SAME tag to resume; a NEW tag for changed settings.
RUN_ROOT = STAGE_ROOT / RUN_TAG
RUN_ROOT.mkdir(parents=True, exist_ok=True)
REPO_URL = 'https://github.com/Rashmika119/Research-project.git'
REPO_REF = 'main'  # Fetch the latest pushed main; record its resolved commit below.
REPO_DIR = Path('/content/Research-project')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
assert not dirty, 'Repository has local edits. Preserve them before selecting the research commit.'
subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
assert Path('run_baseline_fb15k.py').is_file(), 'Select the commit containing this full study.'
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Exact repository SHA:', COMMIT)
LOCK = RUN_ROOT / 'dependencies.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(LOCK if LOCK.exists() else Path('requirements.txt'))], check=True)
# The lock covers packages used by run identity. CUDA/Python are provided by Colab:
# a changed runtime must be made compatible, never silently accepted for resume.
import importlib.metadata as metadata
packages = ('torch', 'torch-geometric', 'transformers', 'huggingface-hub', 'safetensors',
            'numpy', 'matplotlib', 'pandas', 'pyyaml', 'nbformat', 'ipython')
versions = {name: metadata.version(name) for name in packages}
print('Package versions:', versions)
fresh_torch = subprocess.check_output([sys.executable, '-c', 'import torch; print(torch.__version__)'], text=True).strip()
assert str(torch.__version__) == fresh_torch, 'PyTorch changed during installation; restart runtime and rerun setup.'
if not LOCK.exists():
    LOCK.write_text(''.join(f'{name}=={version}\n' for name, version in versions.items()))
print('Persistent dependency lock:', LOCK)


## 3. Approved configuration and full dataset verification


In [ ]:
import json
import yaml
import pandas as pd
from datetime import datetime
from IPython.display import display, Image, Markdown
from training.baseline_fb15k import PROTOCOL, make_config, expected_budget, verify_protocol, verify_manifest
from training.experiment_io import prepare_data, write_json, environment
from training.process import run_logged

EXPERIMENT_DIR = RUN_ROOT / 'runs'
RAW_DIR = SHARED_ROOT / 'data' / 'fb15k'
MANIFEST = EXPERIMENT_DIR / 'dataset_manifest.json'
LOG_DIR = RUN_ROOT / 'logs' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
LOG_DIR.mkdir(parents=True, exist_ok=True)
SKIP_COMPLETED = True
RESUME_INTERRUPTED = True
RUN_STATIC_CHECKS = False
protocol = yaml.safe_load(Path('experiments/configs/fb15k_baseline_300.yaml').read_text(encoding='utf-8'))
verify_protocol(protocol, make_config())
cfg = make_config(RAW_DIR)
display(pd.DataFrame([{'dataset':'FB15K', 'model':'RGAT + ComplEx', 'seed':0,
                      'width':cfg.model['dim'], 'negatives':cfg.num_negatives,
                      'epochs':cfg.epochs, 'warmup':cfg.warmup,
                      'effective_batch':cfg.train_batch_size, 'micro_batch':cfg.micro_batch_size,
                      'accumulation_steps':cfg.accumulation_steps, **expected_budget()}]))
data, graph, known_train, manifest = prepare_data(str(RAW_DIR),0,0,MANIFEST,dataset='fb15k')
verify_manifest(manifest)
assert len(graph.edge_type) == 966284
assert set(data.train) == set(known_train)
print('Full FB15K:', manifest['statistics'])
print('Dataset SHA256:', manifest['dataset_sha256'])
print('Manifest SHA256:', manifest['sha256'])
write_json(RUN_ROOT/'dataset_statistics.json', manifest['statistics'])
write_json(LOG_DIR/'environment.json', environment())
print('One baseline run; no pretrained model or text cache needed.')
print('Persistent outputs:', EXPERIMENT_DIR)
del data, graph, known_train, manifest


## 4. Optional offline CPU checks (no training)


In [ ]:
if RUN_STATIC_CHECKS:
    run_logged([sys.executable,'run_static_checks.py'],LOG_DIR/'static_checks.log')
else:
    print('Optional checks disabled. No preliminary training is launched.')


## 5. Train the existing baseline
Runs one child process. Logs every epoch; validation at10,20,...,300.
Effective batch32768 uses8 scoring micro-batches of4096 and one RGAT encoding.
The final24390 examples use6 micro-batches (5×4096 +3910), weighted by their sizes.
Clipping and Adam run once per effective batch. Full-graph activations still remain resident.
Each completed epoch commits last.pt; best validation saves best.pt.
Resume uses the same tag, source, settings and runtime. If main changes, set
REPO_REF to the recorded original SHA when resuming. Never mix old study outputs.
No automatic batch-size, precision or epoch-budget changes on OOM.
Your compute-unit balance is not a runtime guarantee: use observed epoch timings.


In [ ]:
command = [sys.executable,'-u','run_baseline_fb15k.py',
           '--output-dir',str(EXPERIMENT_DIR),'--raw-dir',str(RAW_DIR)]
if SKIP_COMPLETED: command.append('--skip-completed')
if RESUME_INTERRUPTED: command.append('--resume')
run_logged(command,LOG_DIR/'baseline_training.log')


## 6. Results and progress (also usable after an interruption)


In [ ]:
from training.baseline_fb15k_reports import PLOTS
history_path = EXPERIMENT_DIR/'baseline_seed0/history.json'
if history_path.exists():
    history = json.loads(history_path.read_text(encoding='utf-8'))
    display(pd.DataFrame([{k:v for k,v in h.items() if k not in ('step_losses','validation','learned_gates')}
                          for h in history]).tail(10))
    if history:
        print('Committed progress:', history[-1]['epoch'], '/ 300 epochs;', history[-1]['optimizer_steps'], '/ 4500 updates')
if (EXPERIMENT_DIR/'summary.json').exists():
    display(pd.read_csv(EXPERIMENT_DIR/'metrics.csv'))
    display(Markdown((EXPERIMENT_DIR/'research_analysis.md').read_text(encoding='utf-8')))
    for name in PLOTS:
        display(Image(filename=str(EXPERIMENT_DIR/(name+'.png'))))
else:
    print('No completed report yet; partial histories are not final test results.')


## 7. Regenerate completed reports without training
Can run independently after setting EXPERIMENT_DIR. No GPU or checkpoint loading:
`python run_baseline_fb15k.py --report-only --output-dir /path/to/runs`


In [ ]:
result_path = EXPERIMENT_DIR/'baseline_seed0/results.json'
if result_path.exists():
    from training.baseline_fb15k_reports import baseline_report
    baseline_report(EXPERIMENT_DIR,json.loads(result_path.read_text(encoding='utf-8')))
    print('Reports regenerated:', EXPERIMENT_DIR)
else:
    print('Complete training before generating final reports.')
